<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Modules and imports

**[Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/)** · Python for Practical AI Engineering · Module 3, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** move functions from a script into a package of modules, import them where you need them, and use a main entry point so that importing a file never runs it by accident.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | Nothing to install, and no internet connection: only Python. |
| **You should know** | Functions with a contract, and `count_tickets.py` as functions with a `main()`, from [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/). |
| **You do not need** | Your local project. The setup makes the same `ticket-cleaner` folder here. |
| **Tested** | 2026-10-07, Python 3.14.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M03-L02-modules-and-imports/modules-and-imports-solution.ipynb).

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

You should see `Ready: ticket-cleaner, 1 file`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)


FILES = {
    'count_tickets.py': '''\
TICKETS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]


def filter_by_status(tickets, status="open"):
    """Return the tickets that have the given status."""
    selected = []
    for ticket in tickets:
        if ticket["status"] == status:
            selected.append(ticket)
    return selected


def count_by_category(tickets):
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        category = ticket["category"]
        counts[category] = counts.get(category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets):
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    total = 0
    for ticket in tickets:
        total = total + ticket["priority"]
    return round(total / len(tickets), 1)


def main():
    status = "open"
    selected = filter_by_status(TICKETS, status)
    print(f"Tickets with status '{status}': {len(selected)}")
    if not selected:
        print("No tickets to count.")
        return
    for category, count in count_by_category(selected).items():
        print(f"  {category}: {count}")
    print(f"Average priority: {average_priority(selected)}")


main()
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `tree()` shows the files and folders of the project, like the folder trees on the lesson page.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

def tree():
    """Show the files and folders of the project, one level of indent per folder."""
    print("ticket-cleaner/")
    for path in sorted(PROJECT.rglob("*")):
        parts = path.relative_to(PROJECT).parts
        if any(part in ("__pycache__", ".pytest_cache") for part in parts):
            continue
        print("    " * len(parts) + path.name + ("/" if path.is_dir() else ""))

from pathlib import Path

def check_urgent_report():
    """Check urgent_report.py: the import, the output, and the main entry point."""
    path = Path("urgent_report.py")
    if not path.exists():
        print("Not yet: there is no urgent_report.py. Run the %%writefile cell above first.")
        return
    source = path.read_text(encoding="utf-8")
    if "def filter_by_status" in source:
        print("Not yet: urgent_report.py has its own filter_by_status. Import it from the package instead.")
        return
    if "ticket_cleaner" not in source or "filter_by_status" not in source:
        print("Not yet: import filter_by_status from ticket_cleaner.report.")
        return
    result = run_quiet("urgent_report.py")
    if result.returncode:
        print("Not yet: the script stops with", result.stderr.strip().splitlines()[-1])
        return
    expected = "open: 1 urgent\npending: 0 urgent\nclosed: 1 urgent\n"
    if result.stdout != expected:
        print(f"Not yet: the output is:\n{result.stdout}\nThe task expects:\n{expected}")
        return
    if run_quiet("-c", "import urgent_report").stdout:
        print("Not yet: an import of urgent_report prints its report. Add the main entry point.")
        return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A module is a Python file

A **module** is a file of Python code that other code can import. Its name is the file name without `.py`. The `import` statement runs the module's file once, and then gives you its functions and variables. There are two forms:

- `import platform`, then `platform.python_version()`: the reader sees where the function comes from.
- `from platform import python_version`, then `python_version()`: shorter, for a few names that you use often.

Run the cell. Both lines show the same version.

In [ ]:
import platform
print(platform.python_version())

from platform import python_version
print(python_version())

A **package** is a folder of modules with a file called `__init__.py` in it. You import a module from a package with a dot: `from ticket_cleaner.report import count_by_category`.

## 2. Worked example: split the script

The project now has one file, `count_tickets.py`, with the data, the three functions and `main()`. Run the cell to see the project.

In [ ]:
tree()

> **Check.** You should see `ticket-cleaner/` with `count_tickets.py` and the empty folder `data/`.

The functions move into a package, `ticket_cleaner`, because any program that reports on tickets can use them. The project folder has a `-` in its name, and the package has a `_`: a module or package name must be a valid Python name, and `-` is not allowed in a Python name.

Run the next three cells. They make the package folder with its `__init__.py` (one docstring line), and the module `report.py` with the three functions. The code inside each function does not change.

In [ ]:
from pathlib import Path

Path("ticket_cleaner").mkdir(exist_ok=True)

In [ ]:
%%writefile ticket_cleaner/__init__.py
"""Clean support-ticket records and summarise them."""

In [ ]:
%%writefile ticket_cleaner/report.py
def filter_by_status(tickets, status="open"):
    """Return the tickets that have the given status."""
    selected = []
    for ticket in tickets:
        if ticket["status"] == status:
            selected.append(ticket)
    return selected


def count_by_category(tickets):
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        category = ticket["category"]
        counts[category] = counts.get(category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets):
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    total = 0
    for ticket in tickets:
        total = total + ticket["priority"]
    return round(total / len(tickets), 1)

The script keeps only the part that is specific to this one program: the data and `main()`. It now starts with one import line, and ends with the **main entry point**, which section 4 explains. Run the cell.

In [ ]:
%%writefile count_tickets.py
from ticket_cleaner.report import average_priority, count_by_category, filter_by_status

TICKETS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]


def main():
    status = "open"
    selected = filter_by_status(TICKETS, status)
    print(f"Tickets with status '{status}': {len(selected)}")
    if not selected:
        print("No tickets to count.")
        return
    for category, count in count_by_category(selected).items():
        print(f"  {category}: {count}")
    print(f"Average priority: {average_priority(selected)}")


if __name__ == "__main__":
    main()

Run the script, and show the new layout.

In [ ]:
run("count_tickets.py")
tree()

> **Check.** You should see the same output as before (`Tickets with status 'open': 3` to `Average priority: 2.0`, and `(exit code 0)`), then the folder `ticket_cleaner/` with `__init__.py` and `report.py`. That is the test of a refactor: the structure changed, and the behaviour did not.

On your computer, you also see a folder `__pycache__` in `ticket_cleaner`: Python saves a compiled copy of each imported module there. This notebook turns that off, so that each run reads the newest version of each file.

## 3. How Python finds a module

When Python runs `import ticket_cleaner`, it looks in a list of folders, in order: the folder of the script that you run, then the folders of Python's own modules, then the packages installed in the active environment. The first match wins. So you run `count_tickets.py`, which is next to `ticket_cleaner/`.

## 4. Import runs the file

An import runs every line of the module, once. Each module has a variable `__name__`. When you run a file directly, `__name__` is `"__main__"`. When the file is imported, `__name__` is the module's name.

Run the next two cells. The first makes a small module that prints its name. The second runs it in two ways: as a program (`python ticket_cleaner/name_check.py`), and as an import (`python -c "import ticket_cleaner.name_check"`).

In [ ]:
%%writefile ticket_cleaner/name_check.py
print("name_check.py name:", __name__)

In [ ]:
run("ticket_cleaner/name_check.py")
run("-c", "import ticket_cleaner.name_check")

> **Check.** You should see `name_check.py name: __main__`, then `name_check.py name: ticket_cleaner.name_check`.

This gives you a way to say "run this only when the file is the program, not when it is imported": `if __name__ == "__main__": main()`. Put it at the end of every script. Then a test, or a different script, can import the file to use its functions, without running the whole program.

The notebook itself runs as the main program, as `python file.py` does. Run the cell.

In [ ]:
print("__name__ is:", __name__)


def main():
    print("main() runs")


if __name__ == "__main__":
    main()

> **Check.** You should see `__name__ is: __main__` and `main() runs`.

> **Your turn: change one thing.** Change the condition to `__name__ == "report"`, and run the cell again. Why does `main()` not run now? Change it back.

Delete the small module: it was only to see `__name__`. Run the cell.

In [ ]:
Path("ticket_cleaner/name_check.py").unlink(missing_ok=True)

## 5. Your turn: a second script

The help desk lead wants a short report of all urgent tickets, in any status.

> **Your turn.** Complete `urgent_report.py` in the next cell. It must:
>
> 1. Import `filter_by_status` from the package. Do not copy the function.
> 2. Use the same six tickets (they are in the cell).
> 3. For each status (`"open"`, `"pending"`, `"closed"`), print the status and the number of its tickets with priority `1`, as `open: 1 urgent`.
> 4. End with the main entry point.
>
> Run the cell, then the `run` cell, then the check cell. The check also imports your script, and makes sure that the import prints nothing.

In [ ]:
%%writefile urgent_report.py
from ticket_cleaner.report import filter_by_status

TICKETS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]


def main():
    for status in ["open", "pending", "closed"]:
        selected = filter_by_status(TICKETS, status)
        urgent = [ticket for ticket in selected if ticket["priority"] == 1]
        print(f"{status}: {len(urgent)} urgent")


if __name__ == "__main__":
    main()

In [ ]:
run("urgent_report.py")

In [ ]:
check_urgent_report()

When it is correct, the output is `open: 1 urgent`, `pending: 0 urgent` and `closed: 1 urgent`. The six tickets are now in two scripts. That is a copy, too: in Module 4, both scripts read the tickets from one data file instead.

## 6. Failure case: a file name that hides a Python module

Tomás makes a file called `csv.py` to practise. Later, code that uses Python's own `csv` module stops working.

> **Predict.** Run the next two cells. The first makes `csv.py` in the project folder. The second runs one line that uses Python's `csv` module, from the project folder. What happens?

In [ ]:
%%writefile csv.py
# Practice notes about CSV files.
print("this is my csv.py")

In [ ]:
run("-c", "import csv; print(csv.DictReader)")

> **Check.** You should see `this is my csv.py`, then `AttributeError: module 'csv' has no attribute 'DictReader'`, with the advice to rename the file, and `(exit code 1)`. Python looks for modules in the folder of the program first, so `import csv` found the file `csv.py` instead of Python's module. Common names that cause this are `csv.py`, `json.py`, `random.py`, `test.py`, `email.py` and `httpx.py`.

**Fix:** rename your file to a name that describes your code. Run the cell: it renames the file, then runs the same line again. You should see `<class 'csv.DictReader'>`.

In [ ]:
Path("csv.py").rename("csv_notes.py")
run("-c", "import csv; print(csv.DictReader)")

> **Warning.** If you run a file that is inside the package, and that file imports from `ticket_cleaner`, you see `ModuleNotFoundError: No module named 'ticket_cleaner'`. Python looks in the folder of the file that you run, which is `ticket_cleaner/` itself. Run scripts in the project folder, or use `-m` with the dotted name: `python -m ticket_cleaner.report`.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/). They count toward your certificate when you are signed in and enrolled. On your computer, make the package in your own project, as in the guided practice of the lesson.

In this lesson you learned that a module is a `.py` file and a package is a folder with `__init__.py`, that an import runs the module once, that Python looks in the script's folder first, and that `if __name__ == "__main__":` stops an import from running a script.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.